# YouTube trending videos: views and engagement

## Summary

From November 14, 2017 to June 14, 2018, 161,470 YouTube trending records across the US, Canada, France, and GB show view counts from 223 to about 424.5 million. This notebook examines how views are associated with likes and comments among valid video-country records. It describes associations, not causes.

**Dependent variable:** `views` on the first recorded trending date.

**Comparison measures:** `likes` and `comment_count`. Category, tags, publishing time, and other factors are outside this notebook's scope.

## Table of contents

1. [Part 1: Import libraries](#part-1-import-libraries)
2. [Part 2: View the data](#part-2-view-the-data)
3. [Part 3: Clean and prepare the data](#part-3-clean-and-prepare-the-data)
4. [Part 4: Further data cleaning](#part-4-further-data-cleaning)
5. [Part 5: Visualizations](#part-5-visualizations)
6. [Part 6: Findings and recommendations](#part-6-findings-and-recommendations)

## Part 1: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno


## Part 2: View the data

Load the CSV, check its dimensions and column types, and preview a few records.

In [16]:
# Option 1: If the 'data' folder is in the same folder as your notebook
youtube_df = pd.read_csv('data/youtube.csv')
youtube_df.head()

,index,video_id,trending_date,title,channel_title,category_id,publish_date,time_frame,published_day_of_week,publish_country,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed
0,0,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,13/11/2017,17:00 to 17:59,Monday,US,SHANtell martin,748374,57527,2966,15954,False,False,False
1,1,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,13/11/2017,7:00 to 7:59,Monday,US,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,False,False,False
2,2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,12/11/2017,19:00 to 19:59,Sunday,US,"racist superman|""rudy""""|""""mancuso""""|""""king""""|""...",3191434,146033,5339,8181,False,False,False
3,3,puqaWrEC7tY,17.14.11,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,13/11/2017,11:00 to 11:59,Monday,US,"rhett and link|""gmm""""|""""good mythical morning""...",343168,10172,666,2146,False,False,False
4,4,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,12/11/2017,18:00 to 18:59,Sunday,US,"ryan|""higa""""|""""higatv""""|""""nigahiga""""|""""i dare ...",2095731,132235,1989,17518,False,False,False


In [17]:
youtube_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 161470 entries, 0 to 161469
Data columns (total 18 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   index                   161470 non-null  int64
 1   video_id                161470 non-null  str  
 2   trending_date           161470 non-null  str  
 3   title                   161470 non-null  str  
 4   channel_title           161470 non-null  str  
 5   category_id             161470 non-null  int64
 6   publish_date            161470 non-null  str  
 7   time_frame              161470 non-null  str  
 8   published_day_of_week   161470 non-null  str  
 9   publish_country         161470 non-null  str  
 10  tags                    161470 non-null  str  
 11  views                   161470 non-null  int64
 12  likes                   161470 non-null  int64
 13  dislikes                161470 non-null  int64
 14  comment_count           161470 non-null  int64
 15  comments_di

## Part 3: Clean and prepare the data

Create a working copy. Check missing and blank values, invalid video IDs, status flags, and full-row duplicates. `#NAME?` is a text placeholder, not a null value. Use Missingno to view missingness.

In [ ]:
print('Missing values per column:')
print(youtube_df.isnull().sum())
print('\nMissing values (%):')
print((youtube_df.isnull().mean() * 100).round(2))
print('\nFull-row duplicates:', youtube_df.duplicated().sum())
print('Duplicate rows ignoring index:', youtube_df.drop(columns='index').duplicated().sum())
print('Invalid video IDs:', (youtube_df['video_id'] == '#NAME?').sum())
print('Comments disabled:', youtube_df['comments_disabled'].sum())
print('Ratings disabled:', youtube_df['ratings_disabled'].sum())

msno.matrix(youtube_df)
plt.show()

## Part 4: Further data cleaning

Create `newyoutube_df`, remove invalid IDs and repeated rows, convert the dates, and keep the first trending record for each video-country pair.

In [ ]:
newyoutube_df = youtube_df.drop(columns='index').drop_duplicates().copy()
newyoutube_df = newyoutube_df[newyoutube_df['video_id'] != '#NAME?'].copy()

newyoutube_df['trending_date'] = pd.to_datetime(
    newyoutube_df['trending_date'], format='%y.%d.%m'
)
newyoutube_df['publish_date'] = pd.to_datetime(
    newyoutube_df['publish_date'], format='%d/%m/%Y'
)

newyoutube_df = newyoutube_df.sort_values('trending_date', kind='stable')
newyoutube_df = newyoutube_df.drop_duplicates(
    subset=['video_id', 'publish_country'], keep='first'
)

likes_df = newyoutube_df[newyoutube_df['ratings_disabled'] == False]
comments_df = newyoutube_df[newyoutube_df['comments_disabled'] == False]

print('First-record video-country pairs:', len(newyoutube_df))
newyoutube_df.head()

## Part 5: Visualizations

Compare views with likes and comments using Spearman correlation and simple charts. Skip records where the corresponding feature is disabled. The log-scale scatterplots leave out zero counts.

In [ ]:
print('Views and likes:', likes_df['views'].corr(likes_df['likes'], method='spearman'))
print('Views and comments:', comments_df['views'].corr(comments_df['comment_count'], method='spearman'))

sns.histplot(np.log1p(newyoutube_df['views']), bins=40)
plt.title('Distribution of views')
plt.xlabel('log(1 + views)')
plt.show()

likes_plot_df = likes_df[likes_df['likes'] > 0]
sns.scatterplot(data=likes_plot_df, x='views', y='likes', alpha=0.3)
plt.xscale('log')
plt.yscale('log')
plt.title('Views and likes')
plt.show()

comments_plot_df = comments_df[comments_df['comment_count'] > 0]
sns.scatterplot(data=comments_plot_df, x='views', y='comment_count', alpha=0.3)
plt.xscale('log')
plt.yscale('log')
plt.title('Views and comments')
plt.show()

## Part 6: Findings and recommendations

### Findings

- Views and likes: report the direction and strength of the observed association.
- Views and comments: report the direction and strength of the observed association.

### Limitations

Note invalid IDs, repeated records, disabled engagement fields, and why the results describe associations rather than causes.

### Recommendations

Suggest only next steps supported by the results. Do not claim that likes or comments cause views to increase.